# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Avinish4945/flyRank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

I rank content based on historical search performance. Pages with high impressions but poor average position are prioritized because improving their ranking may increase clicks.

### Rule

If a page has:

- High search impressions
- Average position greater than 10
- Google Search Console data available

then it receives a higher priority score.

### Reason Codes

| Reason Code | Meaning |
|-------------|---------|
| LOW_RANKING | High impressions but poor search ranking |
| LOW_IMPRESSIONS | Not enough search demand |
| NO_GSC_DATA | Search Console data unavailable |
| REVIEW | Manual review recommended |

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
import duckdb
import os

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL="hf://datasets/FlyRank/internship-warehouse"

fact=f"""
read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
"""

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Ranked Queue

The baseline score uses historical impressions and average search position.

Pages with high impressions and lower rankings receive higher scores because improving their ranking may increase future traffic.

Each row receives:

- Score
- Reason Code
- Action Label

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
query=f"""
SELECT
report_date,
client_hash_id,
content_hash_id,
gsc_impressions,
gsc_avg_position,

CASE
WHEN gsc_impressions>1000 AND gsc_avg_position>10 THEN 100
WHEN gsc_impressions>500 AND gsc_avg_position>10 THEN 80
WHEN gsc_impressions>100 THEN 60
ELSE 20
END AS score,

CASE
WHEN gsc_avg_position>10 THEN 'LOW_RANKING'
ELSE 'REVIEW'
END AS reason_code,

CASE
WHEN gsc_avg_position>10 THEN 'Improve SEO'
ELSE 'Monitor'
END AS action_label

FROM {fact}

WHERE
month='2026-03'
AND gsc_data_available IS TRUE

ORDER BY score DESC
"""

df=con.sql(query).df()
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,score,reason_code,action_label
0,2026-03-02,client_fef1a8f436438636,content_91487e6f59e8e82f,1550,60.867097,100,LOW_RANKING,Improve SEO
1,2026-03-02,client_fef1a8f436438636,content_b9b3d13c3af23b6c,1335,28.216479,100,LOW_RANKING,Improve SEO
2,2026-03-02,client_e5c2aa26a8598242,content_39457d17e716086c,1192,38.722315,100,LOW_RANKING,Improve SEO
3,2026-03-02,client_e5c2aa26a8598242,content_714a5f3b963f83c5,1024,21.335938,100,LOW_RANKING,Improve SEO
4,2026-03-03,client_23a62021009f63c4,content_84ac159666c9d8e3,1476,29.342818,100,LOW_RANKING,Improve SEO


In [8]:
os.makedirs("work/outputs",exist_ok=True)

df.to_csv(
"work/outputs/baseline_action_score.csv",
index=False
)

print("CSV Saved")


CSV Saved


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top 20 Review

The highest-ranked pages were manually reviewed.

Each page was checked for:

- Suggested action
- Reason code
- Confidence
- What could make this recommendation incorrect

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20=df.head(20)

top20[
[
"content_hash_id",
"score",
"reason_code",
"action_label"
]
]

,content_hash_id,score,reason_code,action_label
0,content_91487e6f59e8e82f,100,LOW_RANKING,Improve SEO
1,content_b9b3d13c3af23b6c,100,LOW_RANKING,Improve SEO
2,content_39457d17e716086c,100,LOW_RANKING,Improve SEO
3,content_714a5f3b963f83c5,100,LOW_RANKING,Improve SEO
4,content_84ac159666c9d8e3,100,LOW_RANKING,Improve SEO
5,content_3d48b883ddf4c606,100,LOW_RANKING,Improve SEO
6,content_fa3da71588177d93,100,LOW_RANKING,Improve SEO
7,content_f4282b6a12531f84,100,LOW_RANKING,Improve SEO
8,content_8c96fddcf936bb37,100,LOW_RANKING,Improve SEO
9,content_99e520a8b8ff0e59,100,LOW_RANKING,Improve SEO


1.

Action:
Improve SEO

Reason:
High impressions but poor ranking.

Confidence:
Medium

What would make this wrong?

Seasonal search demand.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some recommendations may be incorrect because the rule only considers impressions and ranking.

Possible issues:

- Seasonal traffic
- Missing analytics data
- Recent content updates

### Leakage Check

This rule does not use:

- Future months
- June 2026
- Label-derived columns
- Product decision flags

Therefore no intentional leakage is present.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
query=f"""
SELECT

COUNT(*) total_rows,

COUNT(*) FILTER(
WHERE gsc_data_available IS TRUE
) available_rows

FROM {fact}

WHERE month='2026-03'
"""

con.sql(query).df()

,total_rows,available_rows
0,9841378,3611061


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.